# Phase 8: Exposure at Default (EAD) Modeling
This notebook implements Exposure at Default (EAD) modeling. EAD estimates the exposure amount (outstanding balance) at the time of default. Target is defined as: `EAD % = (funded_amnt - total_rec_prncp) / funded_amnt` capped to `[0.0, 1.0]`.


In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from ead_model import EADModel


## 1. Load Data Splits
Load the preprocessed default datasets.


In [ ]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


## 2. Train EAD Model
Instantiate and fit the EAD XGBoost model on historical defaults.


In [ ]:
ead_model = EADModel()
metrics = ead_model.fit(train_df, oot_df)
print('Model Evaluation Metrics:')
print(metrics)


## 3. Generate Predictions & Reports
Calculate predictions for the OOT validation set, save the pickled model file, and compile the final PDF model report.


In [ ]:
# Predict on OOT
oot_defaults = oot_df[oot_df['loan_status'].isin(['Charged Off', 'Default'])].copy()
oot_defaults['pred_ead_pct'] = ead_model.predict_ead(oot_defaults)
oot_defaults['actual_ead_pct'] = ead_model.calculate_ead_target(oot_defaults)
print(oot_defaults[['actual_ead_pct', 'pred_ead_pct']].describe())

# Save outputs
ead_model.save_model('outputs/scorecards/ead_model.pkl')
oot_defaults[['id', 'member_id', 'actual_ead_pct', 'pred_ead_pct']].to_csv('outputs/scorecards/ead_predictions.csv', index=False)
ead_model.generate_report(train_df, oot_df, 'outputs/reports/ead_model_report.pdf', metrics)
print('EAD predictions and PDF report generated successfully.')
